In [ ]:
# ============================================================
# NB_TRNSF_SilverSFTP_SalesOrderHeader
# Capa: Silver | Origen: SFTP | Tabla: SalesOrderHeader
# Carga incremental SIN upsert (append de fechas nuevas)
# Todo desde config — sin hardcodeos
# ============================================================
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ============================================================
# IDs de workspace y lakehouses — SIN hardcodear (TP)
# ============================================================
# lakehouse.get() funciona en pipeline y en ejecución manual.
# El pipeline setea lh_silver como defaultLakehouse de nb_silver_SFTP.
_lh_silver  = notebookutils.lakehouse.get('lh_silver')
_lh_bronze  = notebookutils.lakehouse.get('lh_bronze')
_lh_landing = notebookutils.lakehouse.get('lh_landing')
WS_ID       = _lh_silver.workspaceId
ART_SILVER  = _lh_silver.id
ART_BRONZE  = _lh_bronze.id
ART_LANDING = _lh_landing.id


# --- Config desde lh_config ---
config = spark.sql("""
    SELECT *
    FROM lh_config.dbo.origin_bronze_silver
    WHERE origen = 'SFTP'
      AND nombre_tabla = 'SalesOrderHeader'
      AND activo = 1
    LIMIT 1
""").collect()[0]

nombre_tabla = config['nombre_tabla']
origen       = config['origen']
pks          = config['pks'].split(',')
params_inc   = config['parametros_incrementales']   # fecha_archivo
tipo_carga   = config['tipo_carga']

# --- IDs técnicos (inmutables del TP) ---

abfs_silver = (
    f"abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com"
    f"/{ART_SILVER}/Tables/{origen}/{nombre_tabla}"
)

print(f"✅ Config cargada")
print(f"   nombre_tabla : {nombre_tabla}")
print(f"   origen       : {origen}")
print(f"   tipo_carga   : {tipo_carga}")
print(f"   pks          : {pks}")
print(f"   params_inc   : {params_inc}")
print(f"   abfs_silver  : {abfs_silver}")


In [ ]:
# ============================================================
# BONUS: Lógica de Reproceso
# ============================================================
try:
    fecha_inicio_reproceso = notebookutils.notebook.getParameter('fecha_inicio_reproceso', '')
    fecha_fin_reproceso    = notebookutils.notebook.getParameter('fecha_fin_reproceso', '')
except Exception:
    fecha_inicio_reproceso = ''
    fecha_fin_reproceso    = ''

modo_reproceso = bool(fecha_inicio_reproceso and fecha_fin_reproceso)
if modo_reproceso:
    print(f"REPROCESO Silver SOH: {fecha_inicio_reproceso} → {fecha_fin_reproceso}")
    spark.sql(
        "DELETE FROM lh_silver." + origen + "." + nombre_tabla +
        " WHERE " + params_inc + " >= '" + fecha_inicio_reproceso + "'" +
        " AND "   + params_inc + " <= '" + fecha_fin_reproceso    + "'"
    )
    ultimo_silver = fecha_inicio_reproceso
    print("   Silver limpiado para el rango")
else:
    print("Modo incremental normal")


In [ ]:
# ============================================================
# Leer registros nuevos desde Bronze
# Patrón TP: WHERE params_inc > ultimo_valor_incremental
#            ORDER BY params_inc  ← garantiza que LIMIT 1000
#            tome siempre el lote más antiguo pendiente
# Watermark se actualiza con MAX del lote al final del run
# ============================================================
tabla_bronze = f"lh_bronze.{origen}.{nombre_tabla}"

# Detectar watermark actual en Silver
try:
    ultimo_silver = (spark.read.format("delta").load(abfs_silver)
        .agg(F.max(params_inc)).collect()[0][0])
    if ultimo_silver is None:
        ultimo_silver = '00000000'
except:
    ultimo_silver = '00000000'

print(f"   Último en Silver : {ultimo_silver}")

# WHERE > watermark + ORDER BY + LIMIT 1000 (obligatorio TP)
# ORDER BY asegura que LIMIT toma el lote cronológicamente más antiguo
df_bronze = spark.sql(f"""
    SELECT * FROM {tabla_bronze}
    WHERE {params_inc} > '{ultimo_silver}'
    ORDER BY {params_inc}
    LIMIT 1000
""")

cnt = df_bronze.count()
print(f"   Filas nuevas desde Bronze: {cnt}")

if cnt == 0:
    print("ℹ️  Sin datos nuevos — Silver ya actualizado")
    notebookutils.notebook.exit("sin_datos_nuevos")

# MAX del lote actual — para actualizar watermark al final
fecha_max_lote = df_bronze.agg(F.max(params_inc)).collect()[0][0]
print(f"   Fecha máx del lote : {fecha_max_lote}")


In [ ]:
# ============================================================
# Tipado y limpieza
# Decimales: coma → punto
# Fechas: Unix timestamp (segundos) → date
# ============================================================
df_silver = df_bronze

# Decimales con coma → punto
for c in ["TotalDue", "SubTotal", "TaxAmt", "Freight"]:
    df_silver = df_silver.withColumn(
        c, F.regexp_replace(F.col(c), ",", ".")
    )

# Fechas Unix timestamp → date
for c in ["OrderDate", "DueDate", "ShipDate"]:
    df_silver = df_silver.withColumn(
        c, F.to_date(F.from_unixtime(F.col(c).cast("bigint")))
    )

# Tipado final
df_silver = (df_silver
    .withColumn("SalesOrderID",    F.col("SalesOrderID").cast("integer"))
    .withColumn("RevisionNumber",  F.col("RevisionNumber").cast("integer"))
    .withColumn("Status",          F.col("Status").cast("integer"))
    .withColumn("OnlineOrderFlag", F.col("OnlineOrderFlag").cast("boolean"))
    .withColumn("CustomerID",      F.col("CustomerID").cast("integer"))
    .withColumn("SalesPersonID",   F.col("SalesPersonID").cast("integer"))
    .withColumn("TerritoryID",     F.col("TerritoryID").cast("integer"))
    .withColumn("BillToAddressID", F.col("BillToAddressID").cast("integer"))
    .withColumn("ShipToAddressID", F.col("ShipToAddressID").cast("integer"))
    .withColumn("ShipMethodID",    F.col("ShipMethodID").cast("integer"))
    .withColumn("CreditCardID",    F.col("CreditCardID").cast("integer"))
    .withColumn("CurrencyRateID",  F.col("CurrencyRateID").cast("integer"))
    .withColumn("SubTotal",        F.col("SubTotal").cast("decimal(18,4)"))
    .withColumn("TaxAmt",          F.col("TaxAmt").cast("decimal(18,4)"))
    .withColumn("Freight",         F.col("Freight").cast("decimal(18,4)"))
    .withColumn("TotalDue",        F.col("TotalDue").cast("decimal(18,4)"))
    .withColumn("ModifiedDate",    F.col("ModifiedDate").cast("timestamp"))
)

# Limpieza
df_silver = df_silver.dropna(subset=pks).dropDuplicates(pks)

for c in ["SalesOrderNumber","PurchaseOrderNumber","AccountNumber",
          "CreditCardApprovalCode","Comment","rowguid"]:
    if c in df_silver.columns:
        df_silver = df_silver.withColumn(c, F.trim(F.col(c)))

print(f"✅ Filas después limpieza: {df_silver.count()}")
display(df_silver.select("SalesOrderID","OrderDate","TotalDue").limit(5))


In [ ]:
# ============================================================
# Guardar en Silver — append incremental
# ============================================================
df_silver = df_silver.dropDuplicates(pks)

(df_silver.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(abfs_silver))

print(f"✅ Datos guardados en Silver: {abfs_silver}")

# ── Actualizar watermark según TP ────────────────────────────
# "Después de cada carga exitosa, se actualiza con el máximo
#  valor encontrado en esa ejecución." (TP Final, sección 8)
# El watermark de Silver SOH vive en la propia tabla Silver
# (MAX de params_inc) — no en lh_config, ya que Bronze SOH
# es quien mantiene ultimo_valor_incremental en config.
print(f"✅ Watermark: {ultimo_silver} → {fecha_max_lote}")
print(f"   Próximo run procesará fechas > {fecha_max_lote}")


In [ ]:
# ============================================================
# Verificar resultado en Silver
# ============================================================
df_ver = spark.read.format("delta").load(abfs_silver)

display(df_ver
    .groupBy(params_inc)
    .agg(
        F.count("*").alias("cant_filas"),
        F.min("OrderDate").alias("primera_orden"),
        F.max("OrderDate").alias("ultima_orden"),
        F.sum("TotalDue").alias("total_ventas")
    )
    .orderBy(params_inc)
)

print(f"✅ Total filas en Silver ({origen}.{nombre_tabla}): {df_ver.count()}")
